# 06 - Modellvergleich und MModellauswahl protokollieren

Der in Notebook 05 freigegebene Entwicklungsstand wird geladen. Dabei werden Feature-Version, Splitbezug und Integrität der gespeicherten Artefakte erneut geprüft. Der Testsplit darf nicht enthalten sein.

## 1. Freigegebenen Entwicklungsstand laden

`modeling.load_development()` verwendet die Prüfungen aus `features.load_development()`. Damit werden Feature-Hashes, Splitbezug, Freigabe und die NaN-Policy erneut kontrolliert.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        p for p in (here, *here.parents)
        if (p / "notebooks").is_dir()
        and (p / "data").is_dir()
        and (p / "src").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Projektordner mit notebooks/, data/ und src/ fehlt."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tlfs23 import features as f
from tlfs23 import modeling as m
from tlfs23.common import paths, read_json, sha256_file

P = paths(PROJECT_ROOT)

development, X, metadata = m.load_development(PROJECT_ROOT)

print("Feature-Matrix:", X.shape)
print("Testdaten enthalten:", development["split"].eq("test").sum())


Feature-Matrix: (73650, 178)
Testdaten enthalten: 0


## 2. Datenübergabe und Detektionsstatus prüfen

Die Klassifikatoren lernen nur aus Zeilen mit `status == "ok"`, weil nur dort 178 gültige Features vorliegen. **Detektionsausfälle bleiben trotzdem in der Validation enthalten** und zählen bei echten Zeichen als End-to-End-Fehler.

Background bedeutet dabei nicht „keine Hand“. Background-Bilder mit zwei erkannten Händen können als negative Beispiele in den Kategorie-Klassifikator eingehen. Ein akzeptiertes Tamil-Zeichen auf einem Background-Bild zählt später als False Accept.


In [2]:
print("Split / Variante:")
display(
    pd.crosstab(
        development["split"],
        development["variant"],
    )
)

print("\nStatus nach Split:")
display(
    pd.crosstab(
        development["split"],
        development["status"],
    )
)

assert development.loc[
    development["split"].eq("validation"),
    "variant",
].eq("original").all()

Split / Variante:


variant,mild_1,original
split,,
train,10800,50400
validation,0,12450



Status nach Split:


status,incomplete_pair,not_detected,ok
split,,,
train,8898,7688,44614
validation,1821,1513,9116


## 3. Bewertungsregeln vor dem Training festlegen

Das Modell wird anhand des höchsten End-to-End-Macro-F1 auf der unveränderten Validation ausgewählt. Bei Gleichstand wird das kleinere Modellartefakt bevorzugt. Für das ausgewählte Modell wird anschließend ausschließlich auf Validation ein Score-Threshold bestimmt. Ziel sind mindestens 95 % korrekte akzeptierte Zeichen bei mindestens 30 akzeptierten Fällen und höchstens 5 % Background-False-Accepts.


In [3]:
SEED = 42
RF_TREES = 120
TARGET_ACCEPTED_SIGN_ACCURACY = 0.95
MAX_BACKGROUND_FALSE_ACCEPT_RATE = 0.05
MIN_ACCEPTED_SIGNS = 30

EXPERIMENTS = m.default_experiments()

display(pd.DataFrame(EXPERIMENTS))


,name,family,include_augmentation
0,dummy_original,dummy,False
1,logreg_original,logistic_regression,False
2,logreg_augmented,logistic_regression,True
3,rf_original,random_forest,False
4,rf_augmented,random_forest,True


## 4. Experimente ausführen oder vorhandene Ergebnisse laden

Solange der finale Testsplit noch nicht geöffnet wurde, werden die definierten Modelle auf Training und Validation verglichen und mit MLflow protokolliert. Nach Öffnung des Testsets wird kein erneutes Training oder Tuning zugelassen. Bei späteren Ausführungen werden ausschließlich die bereits gespeicherten Validation-Ergebnisse geladen.


In [4]:


lock_path = P["work"] / "07_test_access_lock.json"
selection_path = P["work"] / "modeling" / "selected_model.json"

if lock_path.exists():
    # Test wurde bereits geöffnet -> keine neuen Experimente mehr
    selection = read_json(selection_path)

    experiment_dir = PROJECT_ROOT / selection["source_experiment_dir"]

    comparison = pd.read_csv(
        experiment_dir / "validation_comparison.csv"
    )

    thresholds = pd.read_csv(
        experiment_dir / "threshold_validation.csv"
    )

    print("Finaler Test bereits ausgeführt.")
    print("Vorhandene Validation-Ergebnisse wurden geladen.")

else:
    # Test noch ungeöffnet -> Experimente dürfen ausgeführt werden
    comparison, selection = m.train_experiments(
        PROJECT_ROOT,
        experiments=EXPERIMENTS,
        target_accepted_accuracy=TARGET_ACCEPTED_ACCURACY,
        max_background_false_accept_rate=MAX_BACKGROUND_FALSE_ACCEPT_RATE,
        track=True,
    )

    experiment_dir = PROJECT_ROOT / selection["source_experiment_dir"]

    thresholds = pd.read_csv(
        experiment_dir / "threshold_validation.csv"
    )

    print("Experimente ausgeführt und Modellauswahl gespeichert.")

Finaler Test bereits ausgeführt.
Vorhandene Validation-Ergebnisse wurden geladen.


## 5. Modellvergleich ansehen

`detected_sign_*` bewertet den Klassifikator nur auf echten Zeichen mit vollständiger Zwei-Hand-Detektion. `e2e_*` bewertet dagegen die gesamte Validation, sodass `incomplete_pair` und `not_detected` bei echten Zeichen als Fehler erhalten bleiben.


In [5]:
columns = [
    "experiment",
    "family",
    "training_data",
    "e2e_macro_f1",
    "e2e_sign_accuracy",
    "detected_sign_macro_f1",
    "detected_sign_accuracy",
    "sign_coverage",
    "background_false_accept_rate",
    "two_hand_rate_signs",
    "fit_seconds",
    "model_mib",
]

display(
    comparison[columns]
    .round(4)
)

print("Ausgewähltes Experiment:", selection["experiment"])
print("Modellfamilie:", selection["family"])
print("Trainingsdaten:", selection["training_data"])
print("Validation-Ziel erreicht:", selection["validation_target_met"])
print("Gewählter Threshold:", selection["threshold"])


,experiment,family,training_data,e2e_macro_f1,e2e_sign_accuracy,detected_sign_macro_f1,detected_sign_accuracy,sign_coverage,background_false_accept_rate,two_hand_rate_signs,fit_seconds,model_mib
0,rf_augmented,random_forest,original+mild_1,0.7725,0.6760,0.9153,0.9187,0.7342,0.0,0.7359,51.7734,30.0052
1,rf_original,random_forest,original,0.7648,0.6692,0.9058,0.9094,0.7339,0.0,0.7359,40.0128,26.7915
2,logreg_original,logistic_regression,original,0.6354,0.5571,0.7508,0.7570,0.7321,0.0,0.7359,5.5086,0.0350
3,logreg_augmented,logistic_regression,original+mild_1,0.5972,0.5243,0.7051,0.7125,0.7330,0.0,0.7359,8.5822,0.0350
4,dummy_original,dummy,original,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.7359,2.4785,0.0036


Ausgewähltes Experiment: rf_augmented
Modellfamilie: random_forest
Trainingsdaten: original+mild_1
Validation-Ziel erreicht: True
Gewählter Threshold: 0.5


## 7. Threshold-Trade-off nachvollziehen

Ein höherer Threshold kann falsche akzeptierte Zeichen reduzieren, senkt aber typischerweise die Coverage. Background-Bilder dürfen Hände enthalten; als False Accept zählt nur ein **akzeptiertes Tamil-Zeichen** auf einem Background-Eingang.


In [6]:
thresholds = pd.read_csv(
    PROJECT_ROOT / selection["threshold_table_path"]
)

display(
    thresholds[
        [
            "threshold",
            "accepted_sign_accuracy",
            "sign_coverage",
            "background_false_accept_rate",
            "e2e_macro_f1",
            "n_accepted_signs",
        ]
    ].round(4)
)


,threshold,accepted_sign_accuracy,sign_coverage,background_false_accept_rate,e2e_macro_f1,n_accepted_signs
0,0.00,0.9208,0.7342,0.0,0.7725,9067
1,0.30,0.9303,0.7212,0.0,0.7720,8907
2,0.40,0.9391,0.7032,0.0,0.7674,8685
3,0.50,0.9560,0.6674,0.0,0.7551,8243
4,0.60,0.9772,0.6107,0.0,0.7254,7542
5,0.70,0.9892,0.5377,0.0,0.6676,6641
6,0.80,0.9941,0.4395,0.0,0.5738,5428
7,0.90,0.9971,0.2750,0.0,0.3936,3396
8,0.95,1.0000,0.1522,0.0,0.2393,1880


## 8. MLflow und Artefakte

MLflow speichert die Experimente lokal. Zusätzlich werden Modellartefakte, Validation-Predictions, Code-/Daten-Fingerprints, Git-Status und Laufzeitinformationen im Projektordner abgelegt.


In [7]:
print("MLflow Run:", selection["mlflow_run_id"])
print("Modelldatei:", PROJECT_ROOT / selection["model_path"])
print(
    "Auswahlprotokoll:",
    P["work"] / "modeling" / "selected_model.json",
)

MLflow Run: f735d557dd104b6f92ec5f0b4f145b06
Modelldatei: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops/artifacts/tlfs23/modeling/experiments/20261003_190248_9c16aa/rf_augmented/model.joblib
Auswahlprotokoll: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops/artifacts/tlfs23/modeling/selected_model.json


## 9. Modellentscheidung vor Notebook 07 festhalten

Vor dem ersten Zugriff auf den Testsplit wird die Modellentscheidung anhand der Validation eingefroren. Nach der finalen Testauswertung darf der Holdout nicht zum weiteren Tuning verwendet werden.


In [8]:
decision_path = P["work"] / "06_model_decision.json"
selection_path = P["work"] / "modeling" / "selected_model.json"

if decision_path.exists():
    # Bereits festgeschriebene Entscheidung unverändert weiterverwenden
    decision = read_json(decision_path)

    if decision["selection_sha256"] != sha256_file(selection_path):
        raise ValueError(
            "Gespeicherte Modellentscheidung passt nicht zur aktuellen Modellauswahl."
        )

    print("Bereits festgeschriebene Modellentscheidung geladen.")

else:
    DECISION_NOTE = (
    f"{selection['experiment']} wurde ausgewählt, da es mit "
    f"{selected_result['e2e_macro_f1']:.4f} den höchsten End-to-End Macro-F1 "
    f"auf der Validation erreichte. Der Threshold {selection['threshold']:.2f} "
    f"erreicht eine Accepted-Sign-Accuracy von "
    f"{selected_threshold['accepted_sign_accuracy']:.4f} bei einer "
    f"Sign-Coverage von {selected_threshold['sign_coverage']:.4f}. "
    "Die Auswahl erfolgte ausschließlich auf Training und Validation."
)

    decision = m.save_model_decision(
        PROJECT_ROOT,
        DECISION_NOTE,
    )

    print("Modellentscheidung festgeschrieben.")

display(pd.DataFrame([decision]))

Bereits festgeschriebene Modellentscheidung geladen.


,model_version,experiment,family,training_data,threshold,validation_target_met,note,test_used_for_selection,selection_sha256
0,20261003_190248_9c16aa_rf_augmented,rf_augmented,random_forest,original+mild_1,0.5,True,Der Random Forest mit mild augmentierten Train...,False,5f2dfb691c84aa8e835bd75396b669b6572b0069913522...


## Ergebnis

Der Random Forest mit mild_1-augmentierten Trainingsdaten erzielte mit einem End-to-End Macro-F1 von 0,7725 die beste Validation-Leistung. Gegenüber dem Random Forest auf Originaldaten führte die Augmentierung zu einer leichten Verbesserung, während sie bei der logistischen Regression keinen Vorteil zeigte.
Für das ausgewählte Modell wurde auf der Validation ein Threshold von 0,5 festgelegt. Damit werden 95,60 % der akzeptierten Zeichen korrekt bei einer Sign-Coverage von 66,74 % und einer Background-False-Accept-Rate von 0 %. Modell und Threshold wurden vor Öffnung des Testsets eingefroren.
